# Convert `mesolitica/malaysian-whisper-tiny` → sherpa-onnx ONNX

Run this on **Google Colab** (CPU runtime is fine; GPU optional). It:
1. Downloads the Hugging Face model (safetensors).
2. Converts it to **OpenAI Whisper** format (what sherpa-onnx's exporter needs).
3. Runs sherpa-onnx's `export-onnx.py` to produce encoder/decoder ONNX + int8 + tokens.
4. Zips the result for download.

Hand these back for the app:
`tiny-encoder.int8.onnx`, `tiny-decoder.int8.onnx`, `tiny-tokens.txt`.

> ⚠️ **License:** the model card lists no explicit license. Confirm usage terms with mesolitica before shipping/citing in the report.

If any cell errors, copy the full traceback back to me and I'll adjust.

In [ ]:
# 1) Dependencies. We read the weights straight from safetensors, so we DON'T import
#    the transformers model classes (their latest build currently breaks on Colab).
#    onnxscript is required by newer torch's onnx exporter.
!pip -q install -U openai-whisper onnx onnxruntime onnxscript huggingface_hub safetensors
import torch, whisper
print('torch', torch.__version__, '| whisper ok')

In [ ]:
# 2) Read weights from safetensors (no transformers) and remap to OpenAI-Whisper keys.
import torch, dataclasses, json
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from whisper.model import Whisper, ModelDimensions

HF_ID = 'mesolitica/malaysian-whisper-tiny'
sf_path = hf_hub_download(HF_ID, 'model.safetensors')
cfg = json.load(open(hf_hub_download(HF_ID, 'config.json')))
raw = load_file(sf_path)
# WhisperForConditionalGeneration prefixes everything with 'model.'; drop that and
# the tied LM head (proj_out) which OpenAI reuses from the token embedding.
sd = {k[len('model.'):]: v for k, v in raw.items()
      if k.startswith('model.') and not k.startswith('proj_out')}

def rename(k):
    k = k.replace('self_attn.q_proj', 'attn.query')
    k = k.replace('self_attn.k_proj', 'attn.key')
    k = k.replace('self_attn.v_proj', 'attn.value')
    k = k.replace('self_attn.out_proj', 'attn.out')
    k = k.replace('self_attn_layer_norm', 'attn_ln')
    k = k.replace('encoder_attn.q_proj', 'cross_attn.query')
    k = k.replace('encoder_attn.k_proj', 'cross_attn.key')
    k = k.replace('encoder_attn.v_proj', 'cross_attn.value')
    k = k.replace('encoder_attn.out_proj', 'cross_attn.out')
    k = k.replace('encoder_attn_layer_norm', 'cross_attn_ln')
    k = k.replace('fc1', 'mlp.0')
    k = k.replace('fc2', 'mlp.2')
    k = k.replace('final_layer_norm', 'mlp_ln')  # per-layer (top-level handled below)
    k = k.replace('layers', 'blocks')
    return k

new = {}
for key, val in sd.items():
    if key == 'encoder.embed_positions.weight':
        new['encoder.positional_embedding'] = val
    elif key == 'decoder.embed_positions.weight':
        new['decoder.positional_embedding'] = val
    elif key == 'decoder.embed_tokens.weight':
        new['decoder.token_embedding.weight'] = val
    elif key in ('encoder.layer_norm.weight', 'encoder.layer_norm.bias'):
        new[key.replace('encoder.layer_norm', 'encoder.ln_post')] = val
    elif key in ('decoder.layer_norm.weight', 'decoder.layer_norm.bias'):
        new[key.replace('decoder.layer_norm', 'decoder.ln')] = val
    elif key.startswith('encoder.conv1') or key.startswith('encoder.conv2'):
        new[key] = val  # conv names identical in both formats
    else:
        new[rename(key)] = val

dims = ModelDimensions(
    n_mels=cfg['num_mel_bins'],
    n_audio_ctx=cfg['max_source_positions'],
    n_audio_state=cfg['d_model'],
    n_audio_head=cfg['encoder_attention_heads'],
    n_audio_layer=cfg['encoder_layers'],
    n_vocab=cfg['vocab_size'],
    n_text_ctx=cfg['max_target_positions'],
    n_text_state=cfg['d_model'],
    n_text_head=cfg['decoder_attention_heads'],
    n_text_layer=cfg['decoder_layers'],
)
model = Whisper(dims)
missing, unexpected = model.load_state_dict(new, strict=False)
print('MISSING (ok if empty or only *.alignment_heads):', missing)
print('UNEXPECTED (should be empty):', unexpected)
assert not unexpected, 'Key mapping produced unexpected keys — send me this output.'

torch.save({'dims': dataclasses.asdict(dims), 'model_state_dict': model.state_dict()}, 'tiny.pt')
print('saved tiny.pt | n_mels', dims.n_mels, '| n_vocab', dims.n_vocab)

In [ ]:
# 3) Smoke test: load our checkpoint the OpenAI way and transcribe a short clip.
#    (Optional — skip if you have no sample. Records nothing; uses a HF sample.)
m = whisper.load_model('./tiny.pt')
print('loaded ok; params =', sum(p.numel() for p in m.parameters())/1e6, 'M')
# Quick Malay/English sanity check if you upload a 16kHz wav named sample.wav:
import os
if os.path.exists('sample.wav'):
    print(m.transcribe('sample.wav'))
else:
    print('No sample.wav uploaded — skipping transcription test.')

In [ ]:
# 4) Export to sherpa-onnx, forcing the LEGACY onnx exporter (the new dynamo one
#    chokes on Whisper's data-dependent decoder slice). Idempotent + interop fix.
import os, pathlib, whisper, runpy, sys, torch
if not os.path.isdir('sherpa-onnx'):
    os.system('git clone --depth 1 https://github.com/k2-fsa/sherpa-onnx')
p = pathlib.Path('sherpa-onnx/scripts/whisper/export-onnx.py')
p.write_text(p.read_text().replace('torch.set_num_interop_threads(1)', 'pass  # patched'))

if not hasattr(torch.onnx, '_orig_export'):
    torch.onnx._orig_export = torch.onnx.export
def _export_legacy(*a, **k):
    k['dynamo'] = False            # use the TorchScript exporter sherpa expects
    return torch.onnx._orig_export(*a, **k)
torch.onnx.export = _export_legacy

if not hasattr(whisper, '_true_load_model'):
    whisper._true_load_model = whisper.load_model
def _patched_load(name, *a, **k):
    real = whisper._true_load_model
    return real('./tiny.pt') if str(name) == 'tiny' else real(name, *a, **k)
whisper.load_model = _patched_load

sys.argv = ['export-onnx.py', '--model', 'tiny']
try:
    runpy.run_path('sherpa-onnx/scripts/whisper/export-onnx.py', run_name='__main__')
finally:
    whisper.load_model = whisper._true_load_model
    torch.onnx.export = torch.onnx._orig_export
print('\n--- exported files ---')
!ls -lh tiny-*.onnx tiny-tokens.txt

In [ ]:
# 5) Zip the 3 files the app needs + download. (int8 encoder/decoder + tokens.)
import zipfile, os
need = ['tiny-encoder.int8.onnx', 'tiny-decoder.int8.onnx', 'tiny-tokens.txt']
missing = [f for f in need if not os.path.exists(f)]
assert not missing, f'missing: {missing}'
with zipfile.ZipFile('maya-whisper-tiny.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in need:
        z.write(f)
print('zipped:', need, '->', round(os.path.getsize('maya-whisper-tiny.zip')/1e6, 2), 'MB')
from google.colab import files
files.download('maya-whisper-tiny.zip')

## Next steps (app side)
Send me the zip (or just `tiny-encoder.int8.onnx`, `tiny-decoder.int8.onnx`, `tiny-tokens.txt`). Then I'll:
- add the `sherpa_onnx` Flutter package,
- bundle the three files,
- swap the Malay STT path to an offline sherpa-onnx Whisper recognizer (works with **no** Google speech services — ideal for the Huawei),
- keep the device recognizer for English (or use Whisper for both — your call).

Note: Whisper transcribes the whole clip after you stop, so the live word-by-word transcript won't stream (the final text appears at the end).